# Fase 4: Generación de la imagen del dinosaurio (Google Colab)

Modelo text-to-image: **Stable Diffusion XL base 1.0** (`stabilityai/stable-diffusion-xl-base-1.0`), elegido por su calidad; en fp16 cabe en la GPU T4 gratuita de Colab (~7 GB de VRAM). Tarda unos 30-50 s por imagen. `amused-512` (muy rápido pero de calidad baja) queda como alternativa: `MODEL_KEY = 'amused'`.

Qué hace este notebook:
1. Carga la identidad del dinosaurio (generada con Ollama en la Fase 3).
2. Redacta un prompt conciso con el nombre y los rasgos.
3. Genera 4 candidatas (semillas distintas), usted elige la que mejor corresponde a la descripción y se guarda **una sola imagen** por dinosaurio.
4. Levanta una pequeña API (Flask) y la expone con **ngrok (cuenta #2)** para que la aplicación web pida imágenes.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors flask pyngrok

In [ ]:
import os, re, json, time, base64, io, zipfile, shutil
from pathlib import Path
import torch

BASE = Path.cwd() if Path.cwd().name == 'Dinosaurio' else Path.cwd() / 'Dinosaurio'
for sub in ['image_gen/imagenes', 'docs/evidencias']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)
os.chdir(BASE)

assert torch.cuda.is_available(), 'Active la GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4)'
print('GPU:', torch.cuda.get_device_name(0), '| directorio de trabajo:', Path.cwd())

## 1. Módulo del generador y de la API
Se escribe como archivo del repositorio (`image_gen/image_server.py`); el mismo código lo reutiliza la aplicación en la Fase 5.

In [ ]:
%%writefile image_gen/image_server.py
"""Generacion de imagenes (SDXL por defecto, amused como alternativa rapida) y API Flask para la aplicacion web.

Se usa en el notebook de Colab (Fase 4) y en la integracion (Fase 5).
Contrato de la API:
  GET  /health    -> {"status": "ok", "model": ..., "model_key": ..., "device": ...}
  POST /generate  <- {"prompt": str, "negative_prompt"?: str, "seed"?: int, "steps"?: int, "guidance"?: float, "size"?: int}
                  -> {"image_base64": str (PNG), "mime": "image/png", "width", "height", "seed", "steps", "prompt", "seconds"}
"""
import base64
import io
import threading
import time
import urllib.request

MODELS = {
    # Calidad alta, cabe en una T4 (fp16, ~7 GB de VRAM). ~30-50 s por imagen de 1024x1024.
    "sdxl": dict(id="stabilityai/stable-diffusion-xl-base-1.0", steps=30, guidance=7.0, size=1024, max_size=1024),
    # Muy rapido (~3 s) pero de calidad baja: queda como alternativa.
    "amused": dict(id="amused/amused-512", steps=12, guidance=10.0, size=512, max_size=512),
}
DEFAULT_MODEL = "sdxl"
DEFAULT_NEGATIVE = ("cartoon, anime, 3d render, blurry, lowres, deformed, extra limbs, extra heads, "
                    "text, watermark, human")


def load_pipeline(model_key=DEFAULT_MODEL):
    import torch

    cfg = MODELS[model_key]
    if model_key == "sdxl":
        from diffusers import AutoencoderKL, DPMSolverMultistepScheduler, StableDiffusionXLPipeline

        if not torch.cuda.is_available():
            raise RuntimeError("SDXL necesita GPU. Active la GPU T4 en Colab.")
        vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
        pipe = StableDiffusionXLPipeline.from_pretrained(
            cfg["id"], vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
        pipe = pipe.to("cuda")
    else:
        from diffusers import AmusedPipeline

        if torch.cuda.is_available():
            pipe = AmusedPipeline.from_pretrained(cfg["id"], variant="fp16", torch_dtype=torch.float16)
            pipe.vqvae.to(torch.float32)      # el vqvae produce NaN en fp16 (recomendacion del autor)
            pipe = pipe.to("cuda")
        else:
            pipe = AmusedPipeline.from_pretrained(cfg["id"]).to("cpu")
    pipe.dino_model_key = model_key
    return pipe


def _cfg(pipe):
    return MODELS[getattr(pipe, "dino_model_key", DEFAULT_MODEL)]


def generate_image(pipe, prompt, negative_prompt=DEFAULT_NEGATIVE, seed=0, steps=None, guidance=None, size=None):
    import torch

    cfg = _cfg(pipe)
    steps = int(steps or cfg["steps"])
    guidance = float(cfg["guidance"] if guidance is None else guidance)
    size = int(size or cfg["size"])
    gen = torch.Generator(device=pipe.device.type).manual_seed(int(seed))
    out = pipe(prompt, negative_prompt=negative_prompt, num_inference_steps=steps,
               guidance_scale=guidance, generator=gen, height=size, width=size)
    return out.images[0]


def image_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("ascii")


def create_app(pipe):
    from flask import Flask, jsonify, request

    app = Flask(__name__)
    lock = threading.Lock()                    # una generacion a la vez en la GPU
    key = getattr(pipe, "dino_model_key", DEFAULT_MODEL)
    cfg = MODELS[key]

    @app.after_request
    def cors(resp):
        resp.headers["Access-Control-Allow-Origin"] = "*"
        resp.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning"
        resp.headers["Access-Control-Allow-Methods"] = "GET, POST, OPTIONS"
        return resp

    @app.get("/health")
    def health():
        return jsonify(status="ok", model=cfg["id"], model_key=key, device=str(pipe.device))

    @app.route("/generate", methods=["POST", "OPTIONS"])
    def generate():
        if request.method == "OPTIONS":
            return ("", 204)
        data = request.get_json(silent=True) or {}
        prompt = str(data.get("prompt", "")).strip()
        if not prompt:
            return jsonify(error="Falta 'prompt'"), 400
        if len(prompt) > 600:
            return jsonify(error="El prompt es demasiado largo (maximo 600 caracteres)"), 400
        try:
            seed = int(data.get("seed", 0))
            steps = max(1, min(50, int(data.get("steps", cfg["steps"]))))
            guidance = float(data.get("guidance", cfg["guidance"]))
            size = int(data.get("size", cfg["size"]))
            size = max(256, min(cfg["max_size"], (size // 64) * 64))   # multiplo de 64 dentro del limite del modelo
            negative = str(data.get("negative_prompt", DEFAULT_NEGATIVE))
            t0 = time.time()
            with lock:
                img = generate_image(pipe, prompt, negative, seed, steps, guidance, size)
            return jsonify(image_base64=image_to_b64(img), mime="image/png", width=img.width, height=img.height,
                           seed=seed, steps=steps, prompt=prompt, seconds=round(time.time() - t0, 2))
        except Exception as e:                  # noqa: BLE001
            return jsonify(error=f"{type(e).__name__}: {e}"), 500

    return app


def start_server(app, port=5000):
    """Arranca Flask en un hilo y espera a que /health responda."""
    t = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=port, use_reloader=False, threaded=True),
                         daemon=True)
    t.start()
    for _ in range(30):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
            return t
        except Exception:                       # noqa: BLE001
            time.sleep(1)
    raise RuntimeError("El servidor Flask no arranco")


def start_tunnel(port, authtoken):
    """Abre el tunel de ngrok (cuenta #2) y devuelve la URL publica https."""
    from pyngrok import ngrok

    ngrok.set_auth_token(authtoken)
    url = ngrok.connect(port, "http").public_url
    return url.replace("http://", "https://")


In [ ]:
import sys
sys.path.insert(0, str(Path.cwd() / 'image_gen'))
from image_server import (MODELS, DEFAULT_NEGATIVE, load_pipeline, generate_image,
                          image_to_b64, create_app, start_server, start_tunnel)
MODEL_KEY = 'sdxl'          # 'sdxl' (calidad) o 'amused' (rápido, calidad baja)
MODEL_ID = MODELS[MODEL_KEY]['id']
print('Modelo:', MODEL_ID, '|', MODELS[MODEL_KEY])

## 2. Identidad del dinosaurio (entrada de esta fase)
Se descarga de su repositorio de GitHub; si no está allí todavía, el notebook le pedirá subir el archivo `identidad_dinosaurio.json`.

In [ ]:
IDENTIDAD_URL = 'https://raw.githubusercontent.com/ALMA3112/Dinosaurio/main/ollama_setup/identidad_dinosaurio.json'
IDENT_PATH = Path('image_gen/identidad_dinosaurio.json')
try:
    import urllib.request
    urllib.request.urlretrieve(IDENTIDAD_URL, IDENT_PATH)
except Exception as e:
    print('No se pudo descargar desde GitHub:', e)
    from google.colab import files
    print('Suba identidad_dinosaurio.json:')
    up = files.upload()
    shutil.move(next(iter(up)), IDENT_PATH)

identidad = json.loads(IDENT_PATH.read_text(encoding='utf-8'))
NOMBRE = identidad['nombre']
print(NOMBRE, '| modelo de texto:', identidad['modelo_ollama'], '\n')
print('Apariencia:', identidad['apariencia'], '\n')
print('Hábitat:', identidad['habitat'])

## 3. Prompt
El prompt va en inglés (el modelo entiende mejor ese idioma) y se mantiene **conciso**: los codificadores de texto CLIP de SDXL leen como máximo 77 tokens. Se pide una vista de cuerpo completo y un estilo de ilustración paleontológica para que el animal salga completo y creíble. Incluye el nombre y los rasgos visuales de la identidad: cuerpo alargado, cola larga y robusta, cresta dorsal prominente, garras largas y afiladas en las patas delanteras, patas traseras cortas y fuertes, dientes curvos y afilados, y hábitat de bosque húmedo y denso cerca del agua.

Si cambia de dinosaurio, reescriba los rasgos para que correspondan a su descripción.

In [ ]:
PROMPT = (f"{NOMBRE}, a carnivorous dinosaur, full body side view, long body and long thick tail, "
          "tall ornate dorsal crest, long sharp claws on the forelimbs, short strong hind legs, curved sharp teeth, "
          "beside a lake in a dense humid Cretaceous forest, highly detailed paleoart, natural lighting")
NEGATIVE = DEFAULT_NEGATIVE
print('PROMPT:', PROMPT)
print('NEGATIVE:', NEGATIVE)

## 4. Cargar el modelo y generar candidatas
La primera vez descarga el modelo (~7 GB, unos 3-5 minutos). Cada imagen tarda unos 30-50 s en la T4; se generan 4 con semillas distintas (en total ~3 minutos) para elegir la que mejor coincide con la descripción.

In [ ]:
import matplotlib.pyplot as plt

pipe = load_pipeline(MODEL_KEY)
n_tok = len(pipe.tokenizer(PROMPT).input_ids)
print('Tokens del prompt:', n_tok, '(máximo 77; si se pasa, acorte el prompt)')

SEEDS = [0, 1, 2, 3]
candidatas = {}
for s in SEEDS:
    t0 = time.time()
    candidatas[s] = generate_image(pipe, PROMPT, NEGATIVE, seed=s)   # pasos, guidance y tamaño: valores del modelo
    print(f'semilla {s}: {time.time() - t0:.1f} s')

fig, axes = plt.subplots(2, 2, figsize=(10, 10.5))
for ax, s in zip(axes.ravel(), SEEDS):
    ax.imshow(candidatas[s]); ax.set_title(f'semilla {s}'); ax.axis('off')
plt.tight_layout()
plt.savefig('docs/evidencias/imagen_candidatas.png', dpi=130)
plt.show()

### 4.1 Elegir y guardar la imagen
Compare las candidatas con la descripción (cuerpo alargado, cola larga y robusta, cresta dorsal prominente, garras largas, patas traseras fuertes, bosque húmedo cerca del agua). Si ninguna sirve, cambie `SEEDS` por otros números (por ejemplo `[4, 5, 6, 7]`) y vuelva a ejecutar la celda anterior. Escriba la semilla elegida en `SEED_ELEGIDO` y ejecute la celda.

In [ ]:
SEED_ELEGIDO = 0      # <-- cambie por 1, 2 o 3 según la imagen que mejor corresponda a la descripción
img = candidatas[SEED_ELEGIDO]
slug = re.sub(r'[^a-z0-9]+', '_', NOMBRE.lower()).strip('_')
img_path = Path(f'image_gen/imagenes/{slug}.png')
img.save(img_path)

meta = {'nombre': NOMBRE, 'archivo': str(img_path), 'modelo': MODEL_ID, 'prompt': PROMPT,
        'negative_prompt': NEGATIVE, 'seed': SEED_ELEGIDO, 'steps': MODELS[MODEL_KEY]['steps'],
        'guidance': MODELS[MODEL_KEY]['guidance'], 'scheduler': 'DPM++ 2M Karras' if MODEL_KEY == 'sdxl' else 'amused',
        'resolucion': f'{img.width}x{img.height}', 'gpu': torch.cuda.get_device_name(0)}
Path('image_gen/imagen_dinosaurio.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
display(img)
print(json.dumps(meta, indent=2, ensure_ascii=False))

## 5. Paquete de resultados para el repositorio
Se hace **antes** de levantar la API para no perder nada si el túnel falla.

In [ ]:
Path('image_gen/requirements.txt').write_text("""torch
diffusers
transformers
accelerate
safetensors
flask
pyngrok
pillow
""", encoding='utf-8')
archivos = [str(img_path), 'image_gen/imagen_dinosaurio.json', 'image_gen/image_server.py',
            'image_gen/requirements.txt', 'docs/evidencias/imagen_candidatas.png']
with zipfile.ZipFile('/content/fase4_resultados.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a)
try:
    from google.colab import files
    files.download('/content/fase4_resultados.zip')
except ImportError:
    print('Creado: /content/fase4_resultados.zip')

## 6. API de imágenes expuesta con ngrok (cuenta #2)

**Cómo guardar el authtoken sin escribirlo en el notebook:** en Colab, panel izquierdo → icono de la llave (**Secretos**) → *Agregar secreto*: nombre `NGROK_AUTHTOKEN_IMAGE`, valor = el authtoken de su **cuenta de ngrok #2** y activar *Acceso desde el cuaderno*. Si no existe el secreto, la celda se lo pedirá por pantalla (no se muestra).

Esta celda deja la API funcionando. **No cierre la pestaña ni detenga la sesión** mientras use la aplicación. La URL cambia cada vez que se reinicia.

In [ ]:
import requests
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN_IMAGE')
except Exception:
    token = getpass('Authtoken de ngrok (cuenta #2): ')

app = create_app(pipe)
start_server(app, port=5000)
IMAGE_API_URL = start_tunnel(5000, token)
del token
print('API de imágenes:', IMAGE_API_URL)

H = {'ngrok-skip-browser-warning': 'true'}
print('health:', requests.get(f'{IMAGE_API_URL}/health', headers=H, timeout=30).json())
r = requests.post(f'{IMAGE_API_URL}/generate', json={'prompt': PROMPT, 'seed': SEED_ELEGIDO},
                  headers=H, timeout=600).json()
print('prueba /generate:', {k: v for k, v in r.items() if k != 'image_base64'})
from PIL import Image
display(Image.open(io.BytesIO(base64.b64decode(r['image_base64']))))
print('\nCopie esta URL como IMAGE_API_URL en su archivo .env (NO la suba a GitHub).')